# Nettoyage du catalogue Steam

Ce notebook part du fichier brut `data/raw/` et produit un fichier propre dans `data/processed/`. Chaque décision que je vais prendre sera justifiée et chiffrée dans le journal de nettoyage, à la fin de ce notebook.

## 1. Chargement du dataset brut

- Je recharge le fichier d'origine, que je ne modifie jamais à la main.
- Je garde le nombre de lignes de départ dans `nb_depart` pour pouvoir chiffrer tout ce que je retire ensuite.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/steam_games.csv")
nb_depart = len(df)
nb_depart

40833

## 2. Décision 1 : ne garder que les jeux

- Je regarde d'abord la nature des produits de la colonne `types`.

In [2]:
df["types"].value_counts(dropna=False)

types
app       38021
bundle     2572
sub         238
NaN           2
Name: count, dtype: int64

- Je regarde ensuite, pour chaque type, combien de dates de sortie manquent.

In [3]:
df.groupby("types", dropna=False)["release_date"].apply(lambda s: s.isna().sum())

types
app        367
bundle    2572
sub        238
NaN          2
Name: release_date, dtype: int64

- Les `bundle` (2 572) et les `sub` (238) n'ont aucune date de sortie : ce sont des lots de jeux, pas des jeux. Ils fausseraient mes questions sur les prix et sur les sorties par mois. Les 2 valeurs manquantes de `types` ne sont pas des jeux non plus.
- Je ne garde donc que les lignes `app` : 2 812 lignes retirées, il en reste 38 021.

In [4]:
nb_avant = len(df)
df = df[df["types"] == "app"].copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

2812 lignes retirées, il reste 38021


## 3. Décision 2 : les jeux sans nom

- Je compte les jeux dont le nom est manquant : il y en a 14 au total.

In [5]:
df["name"].isna().sum()

np.int64(14)

- Une ligne sans nom ne peut pas être identifiée ni analysée, donc je la supprime : il reste 38 007 jeux.

In [6]:
nb_avant = len(df)
df = df.dropna(subset=["name"]).copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

14 lignes retirées, il reste 38007


## 4. Décision 3 : les doublons

- Je teste combien de lignes ont à la fois le même nom et la même date de sortie : il y en a 4, soit 2 paires.
- Un même nom seul ne suffit pas : 91 lignes partagent un nom avec une autre, mais ce sont surtout des jeux différents (par exemple *Alter Ego* en 2010 et en 1986).

In [7]:
df.duplicated(subset=["name", "release_date"], keep=False).sum()

np.int64(4)

- J'affiche les 4 lignes concernées pour vérifier qu'il s'agit bien de doublons.

In [8]:
brut = pd.read_csv("../data/raw/steam_games.csv")
brut = brut[(brut["types"] == "app") & brut["name"].notna()]
doublons = brut[brut.duplicated(subset=["name", "release_date"], keep=False)]
doublons.sort_values("name")[["name", "release_date", "original_price", "developer"]]

,name,release_date,original_price,developer
1587,Batman: Arkham City - Game of the Year Edition,"Sep 7, 2012",$19.99,"Rocksteady Studios,Feral Interactive (Mac)"
17442,Batman: Arkham City - Game of the Year Edition,"Sep 7, 2012",$19.99,"Rocksteady Studios,Feral Interactive (Mac)"
5410,Warhammer Quest,"Jan 7, 2015",$14.99,"Rodeo Games,Twistplay"
19829,Warhammer Quest,"Jan 7, 2015",NaN,"Twistplay,Rodeo Games"


- *Batman: Arkham City - Game of the Year Edition* et *Warhammer Quest* apparaissent chacun deux fois avec la même date : c'est le même jeu listé deux fois.
- Je supprime ces 2 doublons en gardant la première ligne : il reste 38 005 jeux.

In [9]:
nb_avant = len(df)
df = df.drop_duplicates(subset=["name", "release_date"], keep="first").copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

2 lignes retirées, il reste 38005


- Je vérifie qu'il ne reste plus aucun doublon nom + date : le résultat doit être 0.

In [10]:
df.duplicated(subset=["name", "release_date"], keep=False).sum()

np.int64(0)

## 5. Décision 4 : les prix

- Les prix sont écrits en texte (`$19.99`). Je retire le `$` et je tente de convertir en nombre.
- Ce que je n'arrive pas à convertir devient `NaN` (sans planter), et je compte ces cas.

In [11]:
prix_texte = df["original_price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
prix_num = pd.to_numeric(prix_texte, errors="coerce")
non_num = df.loc[prix_num.isna() & df["original_price"].notna(), "original_price"]
len(non_num)

3018

- Je regarde quelles valeurs ne sont pas des prix.

In [12]:
non_num.value_counts().head(10)

original_price
Free              1326
Free To Play       860
Free to Play       529
Demo                32
Free Demo            8
Download Demo        5
DEMO                 4
Free Mod             4
Play for Free!       3
Try Demo             3
Name: count, dtype: int64

- 2 715 valeurs sont des jeux gratuits (`Free`, `Free To Play`, `Free to Play`) : je leur donne le prix 0.
- Les 303 autres sont des démos, des essais ou du texte libre, pas des prix : je les laisse manquantes.
- Je crée une nouvelle colonne `prix` et je garde `original_price` pour pouvoir comparer.

In [13]:
gratuits = ["Free", "Free To Play", "Free to Play"]
df["prix"] = prix_num
df.loc[df["original_price"].isin(gratuits), "prix"] = 0.0

In [14]:
(df["prix"] == 0).sum(), df["prix"].isna().sum()

(np.int64(2715), np.int64(3310))

## 6. Décision 5 : les dates de sortie

- `release_date` est du texte (`May 12, 2016`) : aucun calcul temporel n'est possible dessus.
- Je la convertis en vraie date dans une nouvelle colonne `date_sortie`. Ce qui n'est pas une date complète devient une date manquante (`NaT`).

In [15]:
df["date_sortie"] = pd.to_datetime(df["release_date"], format="%b %d, %Y", errors="coerce")
df["date_sortie"].isna().sum()

np.int64(1943)

- Je sépare les deux causes : les dates absentes au départ et les valeurs qui ne sont pas des dates.

In [16]:
sans_texte = df["release_date"].isna().sum()
pas_une_date = (df["date_sortie"].isna() & df["release_date"].notna()).sum()
sans_texte, pas_une_date

(np.int64(354), np.int64(1589))

- Je regarde ce que contiennent ces valeurs qui ne sont pas des dates.

In [17]:
df.loc[df["date_sortie"].isna() & df["release_date"].notna(), "release_date"].value_counts().head(8)

release_date
2019           296
Coming Soon    134
TBA             74
Summer 2019     68
2020            52
Coming soon     45
TBD             34
Jun 2019        33
Name: count, dtype: int64

- Ces valeurs sont des jeux pas encore sortis ou des dates imprécises. Je garde ces lignes : les analyses sur les prix ou les genres n'ont pas besoin de la date. Seule l'analyse par période les laissera de côté.

In [18]:
df["date_sortie"].min(), df["date_sortie"].max()

(Timestamp('1981-04-22 00:00:00'), Timestamp('2025-04-03 00:00:00'))

## 7. Décision 6 : les valeurs manquantes

- Je regarde d'abord la part de valeurs manquantes de chaque colonne, pour décider colonne par colonne.

In [19]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False)

recent_reviews              92.9
mature_content              92.4
discount_price              69.1
achievements                67.9
minimum_requirements        44.6
recommended_requirements    44.5
desc_snippet                34.7
all_reviews                 25.1
publisher                   13.1
prix                         8.7
original_price               7.9
date_sortie                  5.1
game_details                 1.2
genre                        1.0
release_date                 0.9
developer                    0.8
game_description             0.3
popular_tags                 0.3
name                         0.0
types                        0.0
url                          0.0
languages                    0.0
dtype: float64

### Colonnes supprimées

- `recent_reviews` et `mature_content` sont vides à plus de 92 %, `achievements` à 68 % : trop de manquants pour servir.
- `minimum_requirements`, `recommended_requirements`, `desc_snippet`, `game_description` et `game_details` sont de longs textes que mes cinq questions n'utilisent pas.
- Je supprime ces 8 colonnes.

In [20]:
colonnes_a_supprimer = [
    "recent_reviews", "mature_content", "achievements",
    "minimum_requirements", "recommended_requirements",
    "desc_snippet", "game_description", "game_details",
]
df = df.drop(columns=colonnes_a_supprimer)
df.shape

(38005, 14)

### Lignes sans `genre` ou sans `languages`

- Le genre est la variable centrale de mes questions 1, 2 et 3, et les langues servent à la question 4.
- Seulement 383 jeux n'ont pas de genre et 4 pas de langue, soit environ 1 % : je vais donc supprimer ces lignes.

In [21]:
nb_avant = len(df)
df = df.dropna(subset=["genre", "languages"]).copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

387 lignes retirées, il reste 37618


### Développeur et éditeur inconnus

- Il manque 191 développeurs et 4 713 éditeurs. L'absence est une information (jeux amateurs, petits studios), et je ne peux pas deviner la valeur.
- Je les remplace par "Inconnu" pour ne pas perdre ces lignes.

In [22]:
df["developer"] = df["developer"].fillna("Inconnu")
df["publisher"] = df["publisher"].fillna("Inconnu")
(df["developer"] == "Inconnu").sum(), (df["publisher"] == "Inconnu").sum()

(np.int64(191), np.int64(4713))

## 8. Décision 7 : les réductions

- Je convertis `discount_price` en nombre dans une nouvelle colonne `prix_reduit`, avec la même méthode que pour les prix.

In [23]:
texte = df["discount_price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
df["prix_reduit"] = pd.to_numeric(texte, errors="coerce")
df["prix_reduit"].notna().sum()

np.int64(11637)

- Une vraie réduction doit être **inférieure** au prix d'origine. Je compare les deux colonnes.

In [24]:
a_un_prix_reduit = df["prix_reduit"].notna()
vraie_reduction = a_un_prix_reduit & (df["prix_reduit"] < df["prix"])
incoherent = a_un_prix_reduit & ~vraie_reduction
a_un_prix_reduit.sum(), vraie_reduction.sum(), incoherent.sum()

(np.int64(11637), np.int64(2625), np.int64(9012))

- Ensuite je regarde des exemples de cas incohérents.

In [25]:
df.loc[incoherent, ["name", "prix", "prix_reduit"]].head(8)

,name,prix,prix_reduit
6,Devil May Cry 5,59.99,70.42
7,Human: Fall Flat,14.99,17.58
11,Danganronpa V3: Killing Harmony,39.99,59.97
13,Call of Duty®: Modern Warfare® Remastered,1.02,906.48
19,Halo Wars: Definitive Edition,19.99,122.31
20,Call of Duty®: Black Ops,1.02,906.48
21,Phoenix Wright: Ace Attorney Trilogy / 逆転裁判123...,29.99,38.25
23,Titan Quest Anniversary Edition,19.99,54.97


- Seulement 2 625 jeux ont un vrai prix réduit, soit environ 23 % des 11 637 jeux qui ont un `discount_price`. Pour 9 012 jeux, la valeur est supérieure ou égale au prix d'origine, ou le prix d'origine est inconnu : je ne sais pas ce qu'elle représente, donc je ne l'interprète pas comme une réduction.
- Je crée `en_reduction` : Vrai pour les 2 625 vraies réductions, Faux pour les 25 981 jeux sans `discount_price` (hypothèse : pas de réduction au moment de la collecte), et manquant pour les 9 012 cas douteux.
- Les cas douteux seront exclus de la question 3 et signalés dans les limites de l'analyse.

In [26]:
df["en_reduction"] = pd.Series(pd.NA, index=df.index, dtype="boolean")
df.loc[~a_un_prix_reduit, "en_reduction"] = False
df.loc[vraie_reduction, "en_reduction"] = True
df["en_reduction"].value_counts(dropna=False)

en_reduction
False    25981
<NA>      9012
True      2625
Name: count, dtype: Int64

## 9. Décision 8 : les prix aberrants

- Je regarde les statistiques de `prix` pour repérer les valeurs extrêmes.

In [27]:
df["prix"].describe()

count     34348.000000
mean         54.044589
std        5278.638336
min           0.000000
25%           1.990000
50%           4.990000
75%           9.990000
max      730640.000000
Name: prix, dtype: float64

- J'applique la règle des 1,5 écart interquartile pour mesurer combien de prix sont « hors normes ».

In [28]:
q1 = df["prix"].quantile(0.25)
q3 = df["prix"].quantile(0.75)
borne_haute = q3 + 1.5 * (q3 - q1)
tres_chers = df[df["prix"] > borne_haute]
borne_haute, len(tres_chers)

(np.float64(21.990000000000002), 3083)

- Je regarde les prix supérieurs à 200 $ : s'ils sont réels, ils devraient être variés.

In [29]:
df.loc[df["prix"] > 200, "prix"].value_counts().head(8)

prix
501.87    96
624.74    61
514.39    29
229.48    20
475.60    18
264.82    17
259.54    17
453.42    16
Name: count, dtype: int64

- Je regarde quels jeux se cachent derrière le prix 624,74 $.

In [30]:
df.loc[df["prix"] == 624.74, "name"].head(5)

6782      Crankies Workshop: Bozzbot Assembly
6783     Crankies Workshop: Grizzbot Assembly
10011                       Fitzzle Wise Owls
10153                              Digit Daze
12565                 Fitzzle Majestic Eagles
Name: name, dtype: str

- Je repère aussi les prix écrits sans le signe `$`, qui n'ont pas le même format que les autres.

In [31]:
format_invalide = (
    df["original_price"].notna()
    & ~df["original_price"].str.startswith("$", na=False)
    & df["prix"].notna()
    & (df["prix"] > 0)
)
df.loc[format_invalide, "original_price"].value_counts()

original_price
1.020     47
650560     1
730640     1
Name: count, dtype: int64

- Je mets en manquant les 49 prix écrits sans `$` : leur format est différent et leurs valeurs sont absurdes.
- Je mets aussi en manquant les prix supérieurs à 200 $ qui se répètent à l'identique sur au moins 5 jeux : c'est une valeur de remplissage, pas un vrai prix.
- Je garde les prix supérieurs à 200 $ qui sont rares (logiciels professionnels comme *VEGAS Pro*) : ils sont plausibles.
- Le prix de comparaison étant faux, je remets aussi `en_reduction` à manquant pour ces lignes.

In [32]:
repete = (df["prix"] > 200) & (df["prix"].map(df["prix"].value_counts()) >= 5)
invalide = format_invalide | repete
format_invalide.sum(), repete.sum(), invalide.sum()

(np.int64(49), np.int64(281), np.int64(330))

In [33]:
df.loc[invalide, "prix"] = float("nan")
df.loc[invalide, "en_reduction"] = pd.NA
df["prix"].isna().sum(), df["prix"].max()

(np.int64(3600), np.float64(995.0))

In [34]:
df["en_reduction"].value_counts(dropna=False)

en_reduction
False    25979
<NA>      9295
True      2344
Name: count, dtype: Int64

## 10. Décision 9 : les dates de sortie futures

- Le jeu de données a été collecté vers juin 2019. Un jeu daté après 2019 n'est pas encore sorti : c'est une date annoncée, pas une vraie sortie.
- Je compte ces jeux.

In [35]:
(df["date_sortie"] > "2019-12-31").sum()

np.int64(31)

- Je regarde les plus lointains.

In [36]:
df.loc[df["date_sortie"] > "2019-12-31", ["name", "release_date"]].sort_values("release_date").head(5)

,name,release_date
382,Cyberpunk 2077,"Apr 15, 2020"
36270,A Purrtato Tail - By the Light of the Elderstar,"Apr 15, 2020"
38508,Zoroastra,"Apr 3, 2025"
34328,Cannibal Lottery - Dystopian Visual Novel,"Aug 17, 2020"
33776,DEEP 8,"Aug 4, 2021"


- Je mets ces 31 dates en manquant : elles fausseraient l'analyse par période. Je garde les jeux, car leurs prix et leurs genres restent valides.
- Pour la question sur les sorties par mois, je m'arrêterai à mai 2019, car les mois suivants sont incomplets.

In [37]:
df.loc[df["date_sortie"] > "2019-12-31", "date_sortie"] = pd.NaT
df["date_sortie"].max(), df["date_sortie"].isna().sum()

(Timestamp('2019-12-31 00:00:00'), np.int64(1963))

## 11. Fichier propre

- Je supprime les colonnes brutes remplacées par leur version propre (`release_date`, `original_price`, `discount_price`) et `types`, qui vaut toujours "app" maintenant. Le fichier d'origine reste intact dans `data/raw/`.

In [38]:
df_propre = df.drop(columns=["types", "release_date", "original_price", "discount_price"])
df_propre.shape

(37618, 12)

- Je vérifie ce qu'il reste de manquants. Ils sont volontaires : jeux sans avis, prix inconnus, dates absentes, réductions douteuses.

In [39]:
df_propre.isna().sum()

url                 0
name                0
all_reviews      9424
developer           0
publisher           0
popular_tags       18
languages           0
genre               0
prix             3600
date_sortie      1963
prix_reduit     25981
en_reduction     9295
dtype: int64

- J'enregistre le fichier propre dans `data/processed/`, sans l'index.

In [40]:
df_propre.to_csv("../data/processed/steam_clean.csv", index=False)

- Je relis le fichier en typant dès la lecture, pour vérifier que tout est conservé : les dates en vraies dates et `en_reduction` en booléen avec valeurs manquantes.

In [41]:
verif = pd.read_csv(
    "../data/processed/steam_clean.csv",
    parse_dates=["date_sortie"],
    dtype={"en_reduction": "boolean"},
)
verif.shape, verif["date_sortie"].dtype, verif["en_reduction"].dtype

((37618, 12), dtype('<M8[us]'), BooleanDtype)

## Journal de nettoyage

| Quoi | Pourquoi | Lignes |
|---|---|---|
| Lignes `bundle`, `sub` et `types` manquant supprimées | Lots de jeux ou pages qui ne sont pas des jeux, sans date de sortie | 2 812 |
| Lignes sans `name` supprimées | Jeu non identifiable | 14 |
| Doublons nom + date supprimés | Même jeu listé deux fois | 2 |
| Prix gratuits (`Free`, `Free to Play`) convertis en 0 | Texte sans montant, mais le jeu est gratuit | 2 715 |
| Prix en texte libre (démos, essais) laissés manquants | Ce ne sont pas des prix | 303 |
| `release_date` convertie en vraie date (`date_sortie`) | Le texte ne permet aucun calcul temporel | 36 062 |
| Dates absentes ou incomplètes laissées manquantes | "Coming Soon", "TBA", "Summer 2019"... : pas de date précise | 1 943 |
| 8 colonnes supprimées (`recent_reviews`, `mature_content`, `achievements`, textes longs...) | Plus de 60 % de manquants, ou texte inutile pour mes questions | 38 005  |
| Lignes sans `genre` ou `languages` supprimées | Variables centrales des questions 1 à 4, environ 1 % des lignes | 387 |
| `developer` et `publisher` manquants remplacés par "Inconnu" | L'absence est une information, valeur non devinable | 191 + 4 713 |
| `discount_price` converti en nombre (`prix_reduit`) | Texte avec `$` | 11 637 |
| Colonne `en_reduction` créée (Vrai si prix réduit < prix d'origine) | Seules les vraies réductions comptent | 2 625 |
| Cas douteux laissés manquants dans `en_reduction` | Prix réduit ≥ prix d'origine ou prix inconnu : valeur non interprétable | 9 012 |
| Prix écrits sans `$` mis en manquant (`1.020`, `650560`, `730640`) | Format différent, valeurs absurdes (Call of Duty à 1,02 $) | 49 |
| Prix > 200 $ répétés à l'identique sur au moins 5 jeux mis en manquant | Valeur de remplissage (501,87 $ pour 96 jeux sans rapport) | 281 |
| Prix > 200 $ rares conservés (VEGAS Pro, GameMaker...) | Plausibles, aberrant ne veut pas dire faux | 23 |
| `en_reduction` remis à manquant sur ces lignes | Le prix de comparaison était faux | 330 |
| Dates après 2019 mises en manquant | Jeu collecté vers juin 2019 : dates annoncées, pas de vraies sorties | 31 |
| Colonnes brutes `types`, `release_date`, `original_price`, `discount_price` supprimées | Remplacées par des versions propres, `types` constant | 4 colonnes |

**Bilan : Avant nettoyage on comptait 40 833 lignes × 20 colonnes au départ, puis après le nettoyage on est a 37 618 lignes × 12 colonnes à l'arrivée.**